In [1]:
import logging
from exp.run import ExperimentRun, SummarySectionName
from exp.config import TransformerExperiments, CNNExperiments
logging.basicConfig(level=logging.ERROR)

In [2]:
config = TransformerExperiments()
config = CNNExperiments()
config.debug = False
config.repeats = 1
config.gpu_id = 1
exp = ExperimentRun(config=config)
in_docker=True


In [3]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch built with CUDA version: {torch.version.cuda}")
print(f"CUDA version: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")

PyTorch version: 2.6.0+cu124
PyTorch built with CUDA version: 12.4
CUDA version: True
CUDA device count: 2


In [4]:
model = config.models[0]
optimizer = config.optimisers[-1]
batch = 10
gpu_id = 1
task_id = None

In [5]:
exp.prepare_monte_carlo_experiments_data(number=600)

## Measure Ground Truth and Estimated Memory for Each job

In [6]:
exp.run_group_truth(in_docker=in_docker)

100%|██████████| 600/600 [00:08<00:00, 69.37it/s]


=============== Start massively run for GPU train ======================


100%|██████████| 600/600 [14:15:06<00:00, 85.51s/it]   
0it [00:00, ?it/s]
0it [00:00, ?it/s]


## Estimate Max GPU Memory by DNNmem

In [7]:
est_list = [
    SummarySectionName.DNNmem,
    SummarySectionName.schedtune
]
if isinstance(exp._config, TransformerExperiments):
    est_list.append(SummarySectionName.LLmem)
exp.run_estimation(
    estimators=est_list,
    in_docker=in_docker
)

if isinstance(exp._config, TransformerExperiments):
    exp.verify_llmem_result()



================== Create docker containers ==================


100%|██████████| 600/600 [20:22<00:00,  2.04s/it]


================== Execute docker containers ==================
=============== Start massively run for GPU train ======================


100%|██████████| 600/600 [3:00:39<00:00, 18.07s/it]  
0it [00:00, ?it/s]


================== Statistics ==================
Run(success/total): 1200/1200


## Estimate Max GPU Memory by SchedTune

In [8]:
exp.statistics()
results = exp.to_evaluation_result()

100%|██████████| 614/614 [00:00<00:00, 18272.98it/s]


=============== Statistics for CNN-Exp ==================
train: 614/614
config: 614/614
groundtruth: 599/614
solution: 599/614
schedtune: 600/614
DNNmem: 600/614
LLmem: 0/614


100%|██████████| 614/614 [00:00<00:00, 3746.07it/s]
